<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/dart_inverso/nb01_kepler_solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB01 — Solver de Kepler: Clase `KeplerSolver`
## Benchmarking de Newton-Raphson, Punto Fijo y Laguerre-Conway para Apophis

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Comparar tres métodos iterativos para resolver la ecuación trascendental de Kepler
> $M = E - e\sin E$, aplicados a la órbita del asteroide (99942) Apophis ($e \approx 0.1914$),
> evaluando convergencia, número de iteraciones y costo computacional a lo largo de un período orbital completo.

## 1. Teoría: La Ecuación de Kepler

### 1.1 Planteamiento

En el **problema de los dos cuerpos** (Kepler), el movimiento de un objeto en una órbita elíptica queda
completamente determinado por la **ecuación de Kepler**:

$$\boxed{M = E - e \sin E}$$

donde:
- $M = n(t - t_p)$ es la **anomalía media** ($n = 2\pi/T$ movimiento medio, $t_p$ tiempo de paso por el perigeo)
- $E$ es la **anomalía excéntrica** (variable geométrica ligada a la elipse auxiliar)
- $e$ es la **excentricidad** de la órbita

La anomalía media $M$ crece uniformemente con el tiempo, mientras que $E$ (y la anomalía verdadera $f$)
no lo hacen: el objeto se mueve más rápido cerca del perigeo y más lento cerca del apogeo.

---

### 1.2 ¿Por qué es trascendental?

La ecuación $M = E - e\sin E$ es **trascendental**: combina una variable algebraica ($E$) con una función
trigonométrica de la misma variable ($\sin E$). No existe una expresión de forma cerrada para $E$ dado $M$.
Cualquier estrategia de solución debe ser **iterativa**, lo que plantea preguntas de:

- **Convergencia global**: ¿el método siempre converge para cualquier $M \in [0, 2\pi]$?
- **Velocidad**: ¿cuántas iteraciones requiere hasta alcanzar la tolerancia deseada?
- **Eficiencia**: ¿cuál es el costo computacional por iteración?

---

### 1.3 Los tres métodos iterativos

Definimos $f(E) = E - e\sin E - M$ y buscamos su raíz. Los tres métodos a comparar son:

| Método | Orden de convergencia | Iteración |
|--------|-----------------------|-----------|
| **Newton-Raphson** | Cuadrático | $E_{n+1} = E_n - \dfrac{f(E_n)}{f'(E_n)}$ |
| **Punto Fijo** | Lineal | $E_{n+1} = M + e\sin E_n$ |
| **Laguerre-Conway** | ~5 (en la práctica) | Ver ecuación abajo |

**Newton-Raphson** explícito:

$$E_{n+1} = E_n - \frac{E_n - e\sin E_n - M}{1 - e\cos E_n}$$

**Punto fijo** (reorganización directa de la ecuación de Kepler):

$$E_{n+1} = M + e\sin E_n$$

**Laguerre-Conway** con orden $n=5$:

$$E_{n+1} = E_n - \frac{5\, f(E_n)}{f'(E_n) \pm \sqrt{\left|16\,[f'(E_n)]^2 - 20\, f(E_n)\, f''(E_n)\right|}}$$

donde $f'(E) = 1 - e\cos E$ y $f''(E) = e\sin E$, y el signo $\pm$ se elige para **maximizar** el denominador,
garantizando estabilidad. Este método fue propuesto por Conway (1986) y adaptado del algoritmo de Laguerre
para polinomios.

---

### 1.4 Por qué la convergencia varía con la posición orbital

La sensibilidad de $E$ respecto a $M$ queda dada por la diferenciación implícita de la ecuación de Kepler:

$$\frac{dE}{dM} = \frac{1}{1 - e\cos E}$$

Cerca del **perigeo** ($E \approx 0$): $\dfrac{dE}{dM} \approx \dfrac{1}{1-e}$.  
Para Apophis ($e \approx 0.191$): $\dfrac{dE}{dM}\big|_{\text{perigeo}} \approx 1.24$ — no es extremo, pero el efecto es observable.

Para órbitas muy excéntricas ($e \to 1$, como cometas), $\dfrac{dE}{dM}\big|_{\text{perigeo}} \to \infty$
y el punto fijo **no converge** (tasa de convergencia $= e < 1$ solo si $e < 1$, pero requiere más
iteraciones conforme $e$ aumenta). Laguerre-Conway fue diseñado precisamente para mantener la convergencia
global incluso en este régimen extremo.

In [ ]:
%pip install -Uq pymcel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import pymcel as pc
from pymcel import constantes as const

print('Librerías cargadas correctamente.')

## 2. Clase `KeplerSolver`

Implementamos una clase que encapsula los tres métodos iterativos con una interfaz uniforme.
Cada método de resolución retorna `(E, error, n_iter)` donde:
- `E`: solución de la anomalía excéntrica [rad]
- `error`: residuo final $|E - e\sin E - M|$
- `n_iter`: número de iteraciones realizadas

También se incluye un método `benchmark` para procesar vectores de anomalías medias y un helper
`check_kepler` para verificar la solución externamente.

In [ ]:
def check_kepler(M, e, E):
    """
    Comprueba el residuo de la ecuación de Kepler para una solución dada.

    Parámetros
    ----------
    M : float o array
        Anomalía media [rad].
    e : float
        Excentricidad.
    E : float o array
        Anomalía excéntrica (solución candidata) [rad].

    Retorna
    -------
    residual : float o array
        |E - e*sin(E) - M|
    """
    return np.abs(E - e * np.sin(E) - M)


class KeplerSolver:
    """
    Solucionador de la ecuación de Kepler: M = E - e*sin(E).

    Implementa y compara tres métodos iterativos clásicos:
      - Newton-Raphson  (convergencia cuadrática)
      - Punto Fijo      (convergencia lineal, tasa = e)
      - Laguerre-Conway (convergencia de alto orden, globalmente convergente)

    Parámetros
    ----------
    e : float
        Excentricidad orbital (0 <= e < 1 para órbitas elípticas).
    delta : float, opcional
        Tolerancia de convergencia en el residuo |E - e*sin(E) - M|.
        Por defecto 1e-12.
    max_iter : int, opcional
        Número máximo de iteraciones permitidas. Por defecto 200.
    """

    def __init__(self, e, delta=1e-12, max_iter=200):
        if not (0.0 <= e < 1.0):
            raise ValueError(f'Excentricidad debe estar en [0, 1). Se recibió e={e}')
        self.e = e
        self.delta = delta
        self.max_iter = max_iter

    # ------------------------------------------------------------------
    # Estimación inicial (Danby 1988, ecuación 6.2.22)
    # ------------------------------------------------------------------
    def _E0(self, M):
        """Estimación inicial robusta basada en Danby (1988)."""
        E0 = M + self.e * np.sin(M) / (1.0 - np.sin(M + self.e) + np.sin(M))
        return E0 if np.isfinite(E0) else M

    # ------------------------------------------------------------------
    # 1. Newton-Raphson
    # ------------------------------------------------------------------
    def newton(self, M):
        """
        Resuelve M = E - e*sin(E) con el método de Newton-Raphson.

        Iteración:
            E_{n+1} = E_n - (E_n - e*sin(E_n) - M) / (1 - e*cos(E_n))

        Convergencia cuadrática: el error se eleva al cuadrado en cada paso.

        Parámetros
        ----------
        M : float
            Anomalía media [rad].

        Retorna
        -------
        E : float
            Anomalía excéntrica [rad].
        error : float
            Residuo final |E - e*sin(E) - M|.
        n_iter : int
            Número de iteraciones realizadas.
        """
        e = self.e
        E = self._E0(M)
        error = np.inf
        for n_iter in range(1, self.max_iter + 1):
            f  = E - e * np.sin(E) - M
            fp = 1.0 - e * np.cos(E)
            dE = -f / fp if abs(fp) > 1e-15 else 0.0
            E += dE
            error = abs(E - e * np.sin(E) - M)
            if error < self.delta:
                break
        return E, error, n_iter

    # ------------------------------------------------------------------
    # 2. Punto Fijo
    # ------------------------------------------------------------------
    def punto_fijo(self, M):
        """
        Resuelve M = E - e*sin(E) con el método de Punto Fijo.

        Iteración:
            E_{n+1} = M + e * sin(E_n)

        Convergencia lineal con tasa igual a la excentricidad e.
        Garantizada para e < 1, pero más lenta conforme e -> 1.

        Parámetros
        ----------
        M : float
            Anomalía media [rad].

        Retorna
        -------
        E : float
            Anomalía excéntrica [rad].
        error : float
            Residuo final |E - e*sin(E) - M|.
        n_iter : int
            Número de iteraciones realizadas.
        """
        e = self.e
        E = M  # estimación inicial simple
        error = np.inf
        for n_iter in range(1, self.max_iter + 1):
            E = M + e * np.sin(E)
            error = abs(E - e * np.sin(E) - M)
            if error < self.delta:
                break
        return E, error, n_iter

    # ------------------------------------------------------------------
    # 3. Laguerre-Conway
    # ------------------------------------------------------------------
    def laguerre(self, M, n=5):
        """
        Resuelve M = E - e*sin(E) con el método de Laguerre-Conway.

        Para f(E) = E - e*sin(E) - M:
            f'(E)  = 1 - e*cos(E)
            f''(E) = e*sin(E)

        Iteración de Laguerre de orden n:
            delta_E = n * f / (f' +/- sqrt(|(n-1)^2*(f')^2 - n*(n-1)*f*f''|))
            E_{k+1} = E_k - delta_E

        El signo se elige para maximizar el valor absoluto del denominador.
        Convergencia de orden 5 en la práctica para la ecuación de Kepler.
        Globalmente convergente (Conway 1986).

        Parámetros
        ----------
        M : float
            Anomalía media [rad].
        n : int, opcional
            Orden del método de Laguerre. Por defecto 5.

        Retorna
        -------
        E : float
            Anomalía excéntrica [rad].
        error : float
            Residuo final |E - e*sin(E) - M|.
        n_iter : int
            Número de iteraciones realizadas.
        """
        e = self.e
        E = self._E0(M)
        error = np.inf
        for n_iter in range(1, self.max_iter + 1):
            f   = E - e * np.sin(E) - M
            fp  = 1.0 - e * np.cos(E)
            fpp = e * np.sin(E)

            # Discriminante para Laguerre
            disc = (n - 1)**2 * fp**2 - n * (n - 1) * f * fpp
            sqrt_disc = np.sqrt(abs(disc))

            # Signo que maximiza el denominador
            denom_pos = fp + sqrt_disc
            denom_neg = fp - sqrt_disc
            denom = denom_pos if abs(denom_pos) >= abs(denom_neg) else denom_neg

            if abs(denom) < 1e-15:
                denom = fp  # fallback a Newton

            E -= n * f / denom
            error = abs(E - e * np.sin(E) - M)
            if error < self.delta:
                break
        return E, error, n_iter

    # ------------------------------------------------------------------
    # Dispatcher
    # ------------------------------------------------------------------
    def solve(self, M, method='newton'):
        """
        Resuelve la ecuación de Kepler usando el método especificado.

        Parámetros
        ----------
        M : float
            Anomalía media [rad].
        method : str, opcional
            Método a usar: 'newton', 'punto_fijo', o 'laguerre'.

        Retorna
        -------
        (E, error, n_iter) : tuple
        """
        dispatch = {
            'newton':     self.newton,
            'punto_fijo': self.punto_fijo,
            'laguerre':   self.laguerre,
        }
        if method not in dispatch:
            raise ValueError(f'Método desconocido: {method!r}. '
                             f'Opciones: {list(dispatch.keys())}')
        return dispatch[method](M)

    # ------------------------------------------------------------------
    # Benchmark vectorizado
    # ------------------------------------------------------------------
    def benchmark(self, M_array, method):
        """
        Resuelve la ecuación de Kepler para un vector de anomalías medias
        y registra el tiempo CPU de cada llamada.

        Parámetros
        ----------
        M_array : array_like
            Vector de anomalías medias [rad].
        method : str
            Método a usar: 'newton', 'punto_fijo', o 'laguerre'.

        Retorna
        -------
        E_arr : ndarray
            Anomalías excéntricas solución.
        iters_arr : ndarray (int)
            Número de iteraciones por llamada.
        times_arr : ndarray
            Tiempo CPU por llamada [segundos].
        """
        N = len(M_array)
        E_arr     = np.empty(N)
        iters_arr = np.empty(N, dtype=int)
        times_arr = np.empty(N)

        for i, M in enumerate(M_array):
            t0 = time.perf_counter()
            E, _, n_it = self.solve(M, method=method)
            times_arr[i] = time.perf_counter() - t0
            E_arr[i]     = E
            iters_arr[i] = n_it

        return E_arr, iters_arr, times_arr


# Verificación rápida
print('KeplerSolver y check_kepler definidos correctamente.')
s_test = KeplerSolver(e=0.5, delta=1e-12)
M_t = np.pi / 3
En, errn, nn = s_test.newton(M_t)
Ep, errp, np_ = s_test.punto_fijo(M_t)
El, errl, nl = s_test.laguerre(M_t)
print(f'Test e=0.5, M={M_t:.4f}:')
print(f'  Newton-Raphson : E={En:.10f}  err={errn:.2e}  iter={nn}')
print(f'  Punto Fijo     : E={Ep:.10f}  err={errp:.2e}  iter={np_}')
print(f'  Laguerre-Conway: E={El:.10f}  err={errl:.2e}  iter={nl}')
print(f'  check_kepler   : {check_kepler(M_t, 0.5, En):.2e}')

## 3. Elementos orbitales de Apophis

El asteroide **(99942) Apophis** es uno de los objetos más estudiados de la categoría NEO (*Near-Earth Object*).
Se acercará a la Tierra el **13 de abril de 2029** a ~38 000 km, dentro de la órbita de satélites geoestacionarios.

Los elementos orbitales usados en este notebook corresponden al estado orbital en la época J2000,
obtenidos del catálogo JPL Small-Body Database Browser (SBDB):

| Parámetro | Símbolo | Valor |
|-----------|---------|-------|
| Semieje mayor | $a$ | 0.9223 AU |
| Excentricidad | $e$ | 0.19147 |
| Inclinación | $i$ | 3.337° |
| Período | $T$ | $\approx$ 323.6 días |

In [ ]:
# Elementos orbitales de Apophis (99942) en 2029 (aproximados, epoch J2000)
e_apophis = 0.19147     # excentricidad
a_apophis = 0.9223      # semieje mayor [AU]
T_apophis = a_apophis**(1.5) * 365.25   # período [días], ley de Kepler
n_apophis = 2*np.pi / T_apophis          # movimiento medio [rad/día]

print(f'Apophis (99942):')
print(f'  e = {e_apophis:.5f}')
print(f'  a = {a_apophis:.4f} AU')
print(f'  T = {T_apophis:.2f} días = {T_apophis/365.25:.3f} años')
print(f'  n = {n_apophis:.6f} rad/día')

# Crear el solver para Apophis
solver = KeplerSolver(e=e_apophis, delta=1e-12)
print(f'\nKeplerSolver creado: e={solver.e}, delta={solver.delta}')

## 4. Demostración: Solución para valores representativos de $M$

Evaluamos los tres métodos en cinco valores representativos de la anomalía media:
- $M \approx 0$ (perigeo, zona de mayor dificultad)
- $M = 0.5$ rad (región intermedia)
- $M = \pi/2$ rad (cuadratura)
- $M = \pi$ rad (apogeo, punto más fácil)
- $M = 3\pi/2$ rad (segunda cuadratura)

In [ ]:
M_demo = [0.01, 0.5, np.pi/2, np.pi, 3*np.pi/2]

print(f"{'M [rad]':>12} {'E Newton':>12} {'E Punto Fijo':>14} {'E Laguerre':>12} {'Residuo (N)':>12}")
print("-" * 68)
for M in M_demo:
    En, errn, nn = solver.newton(M)
    Ep, errp, np_ = solver.punto_fijo(M)
    El, errl, nl = solver.laguerre(M)
    res = check_kepler(M, e_apophis, En)
    print(f"{M:12.6f} {En:12.8f} {Ep:14.8f} {El:12.8f} {res:12.2e}")

## 5. Benchmark: Órbita completa

Evaluamos los tres métodos en 1000 valores de $M$ equidistribuidos en $[0, 2\pi)$,
registrando el número de iteraciones y el tiempo CPU de cada llamada.

In [ ]:
M_arr = np.linspace(0.001, 2*np.pi - 0.001, 1000)

methods = ['newton', 'punto_fijo', 'laguerre']
colores = {'newton': 'royalblue', 'punto_fijo': 'tomato', 'laguerre': 'seagreen'}
labels  = {'newton': 'Newton-Raphson', 'punto_fijo': 'Punto Fijo', 'laguerre': 'Laguerre-Conway'}

results = {}
for m in methods:
    E_arr, iters_arr, times_arr = solver.benchmark(M_arr, m)
    results[m] = {'E': E_arr, 'iters': iters_arr, 'times': times_arr}
    print(f"{labels[m]:20s}: iters promedio = {np.mean(iters_arr):.2f}, "
          f"tiempo total = {np.sum(times_arr)*1e6:.1f} μs")

## 6. Visualización del benchmark — 4 paneles

Analizamos cuatro métricas a lo largo de la órbita completa:
1. **Número de iteraciones** vs $M$: ¿dónde trabaja más cada método?
2. **Residuo** $|E - e\sin E - M|$ vs $M$: ¿cuánta precisión alcanza cada método?
3. **Tiempo CPU** por llamada vs $M$: costo computacional.
4. **$E(M)$** con destacado de la región cercana al perigeo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f'Benchmark KeplerSolver — Apophis (99942), $e = {e_apophis}$',
    fontsize=14, fontweight='bold'
)

# ---- Panel 1: Iteraciones vs M ----------------------------------------
ax1 = axes[0, 0]
for m in methods:
    ax1.plot(M_arr, results[m]['iters'],
             color=colores[m], label=labels[m], lw=1.5, alpha=0.85)
ax1.set_xlabel('$M$ [rad]')
ax1.set_ylabel('Iteraciones')
ax1.set_title('Número de iteraciones')
ax1.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax1.set_xticklabels(['0', '$\\pi/2$', '$\\pi$', '$3\\pi/2$', '$2\\pi$'])
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.35)

# ---- Panel 2: Residuo vs M --------------------------------------------
ax2 = axes[0, 1]
for m in methods:
    residuals = check_kepler(M_arr, e_apophis, results[m]['E'])
    ax2.semilogy(M_arr, np.maximum(residuals, 1e-17),
                 color=colores[m], label=labels[m], lw=1.5, alpha=0.85)
ax2.axhline(1e-12, color='gray', ls='--', lw=1, label='Tolerancia $\\delta$')
ax2.set_xlabel('$M$ [rad]')
ax2.set_ylabel('Residuo $|E - e\\sin E - M|$')
ax2.set_title('Residuo de convergencia')
ax2.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax2.set_xticklabels(['0', '$\\pi/2$', '$\\pi$', '$3\\pi/2$', '$2\\pi$'])
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.35)

# ---- Panel 3: Tiempo CPU vs M -----------------------------------------
ax3 = axes[1, 0]
for m in methods:
    ax3.plot(M_arr, results[m]['times'] * 1e6,
             color=colores[m], label=labels[m], lw=1.0, alpha=0.75)
ax3.set_xlabel('$M$ [rad]')
ax3.set_ylabel('Tiempo por llamada [μs]')
ax3.set_title('Costo computacional')
ax3.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax3.set_xticklabels(['0', '$\\pi/2$', '$\\pi$', '$3\\pi/2$', '$2\\pi$'])
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.35)

# ---- Panel 4: E(M) con destacado del perigeo --------------------------
ax4 = axes[1, 1]
# Referencia: E = M (sin excentricidad)
ax4.plot(M_arr, M_arr, color='lightgray', ls='--', lw=1.5, label='$E = M$ (circular)')
ax4.plot(M_arr, results['newton']['E'],
         color=colores['newton'], lw=2, label='$E(M)$ Newton (Apophis)')
# Destacar región del perigeo
mask_peri = M_arr < 0.4
ax4.fill_between(M_arr[mask_peri], M_arr[mask_peri], results['newton']['E'][mask_peri],
                  color='gold', alpha=0.4, label='Zona perigeo ($M < 0.4$)')
ax4.set_xlabel('$M$ [rad]')
ax4.set_ylabel('$E$ [rad]')
ax4.set_title('Anomalía excéntrica $E(M)$')
ax4.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax4.set_xticklabels(['0', '$\\pi/2$', '$\\pi$', '$3\\pi/2$', '$2\\pi$'])
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('benchmark_kepler_apophis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en benchmark_kepler_apophis.png')

## 7. Zoom: Convergencia cerca del perigeo

La región $M \in [0, 0.2]$ rad es donde se esperan las mayores diferencias entre métodos,
porque la derivada $dE/dM = 1/(1-e\cos E)$ es máxima cerca de $E \approx 0$.

Para Apophis: $dE/dM|_{\text{perigeo}} = 1/(1-0.191) \approx 1.236$.

Aunque no es un valor extremo, el número de iteraciones del Punto Fijo debería ser notablemente
mayor que el de Newton-Raphson y Laguerre-Conway en esta región.

In [ ]:
M_zoom = np.linspace(0.001, 0.2, 300)

zoom_results = {}
for m in methods:
    E_z, iters_z, times_z = solver.benchmark(M_zoom, m)
    zoom_results[m] = {'E': E_z, 'iters': iters_z, 'times': times_z}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Zoom: Convergencia en la región del perigeo ($M \\in [0, 0.2]$ rad)',
    fontsize=13, fontweight='bold'
)

# Panel izquierdo: iteraciones en la zona del perigeo
ax = axes[0]
for m in methods:
    ax.plot(M_zoom, zoom_results[m]['iters'],
            color=colores[m], label=labels[m], lw=2)
ax.set_xlabel('$M$ [rad]')
ax.set_ylabel('Iteraciones')
ax.set_title('Iteraciones — zona perigeo')
ax.legend()
ax.grid(True, alpha=0.35)

# Panel derecho: residuos en la zona del perigeo
ax = axes[1]
for m in methods:
    res_z = check_kepler(M_zoom, e_apophis, zoom_results[m]['E'])
    ax.semilogy(M_zoom, np.maximum(res_z, 1e-17),
                color=colores[m], label=labels[m], lw=2)
ax.axhline(1e-12, color='gray', ls='--', lw=1, label='Tolerancia')
ax.set_xlabel('$M$ [rad]')
ax.set_ylabel('Residuo $|E - e\\sin E - M|$')
ax.set_title('Residuo — zona perigeo')
ax.legend()
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

## 8. Tabla comparativa con `pymcel`

Verificamos que los resultados de `KeplerSolver` son consistentes con las implementaciones
de referencia de `pymcel`: `pc.kepler_newton`, `pc.kepler_semianalitico`, `pc.kepler_eserie`
y `pc.metodo_laguerre`.

La función `pc.kepler_newton(M, e)` resuelve la ecuación de Kepler con Newton-Raphson y
retorna `(E, error, n_iter)`, igual que nuestra implementación.

In [ ]:
# Comparación de métodos propios vs pymcel para M seleccionados
M_cmp = np.array([0.01, 0.5, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi - 0.01])
e = e_apophis

print(f'Comparación KeplerSolver vs pymcel  (e = {e:.5f})')
print(f'{"M [rad]":>10}  {"KS Newton":>14}  {"pc.newton":>14}  '
      f'{"KS PF":>10}  {"pc.semian":>10}  {"KS Lag":>12}  {"pc.eserie":>10}')
print('-' * 95)

for M in M_cmp:
    En_ks, _, _ = solver.newton(M)
    Ep_ks, _, _ = solver.punto_fijo(M)
    El_ks, _, _ = solver.laguerre(M)

    En_pc, _, _ = pc.kepler_newton(M, e)
    Esa_pc, _, _ = pc.kepler_semianalitico(M, e)
    Ese_pc, _, _ = pc.kepler_eserie(M, e)

    print(f'{M:10.5f}  {En_ks:14.10f}  {En_pc:14.10f}  '
          f'{Ep_ks:10.8f}  {Esa_pc:10.8f}  {El_ks:12.10f}  {Ese_pc:10.8f}')

## 9. Tabla resumen de resultados

Consolidamos las métricas estadísticas del benchmark sobre la órbita completa.

In [ ]:
print('='*75)
print(f'  RESUMEN BENCHMARK — Apophis (e={e_apophis:.5f}), 1000 puntos en [0.001, 2π-0.001]')
print('='*75)
print(f'  {"Método":<20} {"Iter media":>10} {"Iter max":>9} {"T total [μs]":>14} {"T media [μs]":>13}')
print('-'*75)

for m in methods:
    it  = results[m]['iters']
    tm  = results[m]['times']
    print(f'  {labels[m]:<20} {np.mean(it):>10.2f} {np.max(it):>9d} '
          f'{np.sum(tm)*1e6:>14.1f} {np.mean(tm)*1e6:>13.4f}')

print('='*75)

# Máximo residuo alcanzado por cada método
print()
print(f'  {"Método":<20} {"Residuo max":>14} {"Residuo media":>14}')
print('-'*55)
for m in methods:
    res = check_kepler(M_arr, e_apophis, results[m]['E'])
    print(f'  {labels[m]:<20} {np.max(res):>14.2e} {np.mean(res):>14.2e}')
print('='*55)

## 10. Escalamiento con la excentricidad

Finalmente, exploramos cómo varía el número de iteraciones promedio de cada método
en función de la excentricidad $e \in [0.01, 0.95]$.

Esto permite entender por qué para órbitas muy excéntricas (cometas, asteroides en órbitas
de Kozai) el Punto Fijo falla y Laguerre-Conway sigue siendo la opción más robusta.

In [ ]:
e_vals = np.linspace(0.01, 0.95, 50)
M_test_arr = np.linspace(0.01, 2*np.pi - 0.01, 200)

mean_iters = {m: [] for m in methods}

for e_val in e_vals:
    s = KeplerSolver(e=e_val, delta=1e-10, max_iter=500)
    for m in methods:
        _, it_arr, _ = s.benchmark(M_test_arr, m)
        mean_iters[m].append(np.mean(it_arr))

fig, ax = plt.subplots(figsize=(10, 5))
for m in methods:
    ax.plot(e_vals, mean_iters[m], color=colores[m], label=labels[m], lw=2)

# Marcar posición de Apophis
ax.axvline(e_apophis, color='darkorange', ls=':', lw=2, label=f'Apophis ($e={e_apophis}$)')

ax.set_xlabel('Excentricidad $e$', fontsize=12)
ax.set_ylabel('Iteraciones promedio', fontsize=12)
ax.set_title('Escalamiento de iteraciones con la excentricidad', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

## 11. Conclusiones

### 11.1 Resumen de resultados

| Método | Convergencia | Iteraciones (Apophis) | Observaciones |
|--------|-------------|----------------------|---------------|
| **Newton-Raphson** | Cuadrática | 2–4 | Robusto para la mayoría de órbitas; excelente balance velocidad/precisión |
| **Punto Fijo** | Lineal ($\rho = e$) | 10–20 | Simple de implementar, pero requiere más iteraciones especialmente al aumentar $e$ |
| **Laguerre-Conway** | ~5 (Kepler) | 2–3 | El más robusto; globalmente convergente incluso para $e \to 1$ |

### 11.2 Interpretación para Apophis ($e \approx 0.191$)

- Para $e \approx 0.19$, **los tres métodos convergen bien** en toda la órbita.
- **Newton-Raphson** es la elección natural para este rango de excentricidad: 2–3 iteraciones
  con muy poco overhead por iteración.
- **Punto Fijo** requiere ~10–15 iteraciones, siendo funcional pero ineficiente comparado
  con los otros dos.
- **Laguerre-Conway** converge en el mismo número de iteraciones que Newton-Raphson o menos,
  con la ventaja adicional de ser globalmente convergente.

### 11.3 Generalización a otras órbitas

- Para $e > 0.5$: el **Punto Fijo** empieza a requerir muchas iteraciones y puede ser
  imprácticamente lento cerca del perigeo.
- Para $e > 0.9$ (cometas de período corto): el **Punto Fijo** puede requerir cientos de
  iteraciones; **Newton-Raphson** aún funciona pero puede fallar con estimaciones iniciales
  malas; **Laguerre-Conway** mantiene la convergencia global.
- Para aplicaciones de propagación orbital de alta precisión (e.g., misiones de deflexión
  tipo DART), se recomienda **Laguerre-Conway** como solver base por su robustez.

### 11.4 Relevancia para el proyecto Apophis Inverse DART

En el contexto de la propagación de la órbita post-impacto de Apophis, el solver de Kepler
se invoca miles de veces (una por paso temporal en la integración keplerina). La elección
del método tiene impacto directo en el tiempo de cómputo de las simulaciones de Monte Carlo
sobre el parámetro de deflexión $\Delta v$.

Los notebooks posteriores (`nb02` en adelante) utilizarán la clase `KeplerSolver` con el
método `laguerre` como solver predeterminado.